# 6. Analizzare un documento con un agente MAF

Collega Content Understanding a un agente Microsoft Agent Framework.

## 1. Import e configurazione

Carica librerie, variabili di ambiente e documento.

In [ ]:
import os
from pathlib import Path

from agent_framework import Agent, Content, Message
from agent_framework.foundry import ContentUnderstandingContextProvider, FoundryChatClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [ ]:
load_dotenv(r"..\.env", override=True)


In [ ]:
DOCUMENTO = Path(r"..\file\esercizio1.pdf")
document_bytes = DOCUMENTO.read_bytes()


## 2. Content Understanding come context provider

Configura l'analisi del PDF come contesto dell'agente.

In [ ]:
credential = DefaultAzureCredential()


In [ ]:
content_understanding = ContentUnderstandingContextProvider(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"],
    credential=credential,
    analyzer_id="prebuilt-documentSearch",
    max_wait=None,
)


In [ ]:
chat_client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_ENDPOINT"],
    model="gpt-5.4",
    credential=credential,
)


## 3. Agente ed esecuzione

Crea l'agente Foundry e mostra la risposta basata sul documento.

In [ ]:
agente = Agent(
    client=chat_client,
    name="AnalistaDocumenti",
    instructions="Analizza i documenti e rispondi in italiano in modo chiaro e sintetico.",
    context_providers=[content_understanding],
)


In [ ]:
richiesta = Message(
    role="user",
    contents=[
        Content.from_text(
            "Analizza il documento, riassumilo ed elenca le informazioni principali."
        ),
        Content.from_data(
            document_bytes,
            "application/pdf",
            additional_properties={"filename": DOCUMENTO.name},
        ),
    ],
)


In [ ]:
try:
    async with content_understanding:
        risposta = await agente.run(richiesta)
finally:
    credential.close()


In [ ]:
print(risposta.text)
